# PANGAEA 00 Load

Inspect raw files for the PANGAEA pipeline and save an explicit dataset summary under `results/pangaea/`.


In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from src.config import get_dataset_config
from src.datasets.pangaea import build_pangaea_summary
from src.io.pangaea import load_pangaea_dataset
from src.utils.plotting import plot_signal_panel

CONFIG = get_dataset_config("pangaea")
RESULTS_DIR = CONFIG.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
summary = build_pangaea_summary()
summary_path = RESULTS_DIR / "dataset_summary.json"
inventory_path = RESULTS_DIR / "raw_file_inventory.json"
readme_path = RESULTS_DIR / "README.md"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
readme_path.write_text(
    "\n".join(
        [
            "# PANGAEA Dataset Summary",
            "",
            f"- Source URL: {summary['dataset']['source_url']}",
            f"- Analysis ready: {summary.get('analysis_ready', False)}",
            f"- Notes: {summary.get('notes', summary['dataset'].get('notes', ''))}",
            f"- Raw file: {summary.get('raw_file')}",
        ]
    )
    + "\n",
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))
inventory = {
    "dataset": "pangaea",
    "raw_dir": str(CONFIG.raw_dir),
    "filenames_found": sorted(path.name for path in CONFIG.raw_dir.iterdir() if path.is_file()),
    "file_types_found": sorted({path.suffix.lower() or "<no_extension>" for path in CONFIG.raw_dir.iterdir() if path.is_file()}),
    "loader": "load_pangaea_dataset",
    "loader_status": "ok" if summary.get("raw_file") and not summary.get("error") else "error",
    "loader_raw_file": summary.get("raw_file"),
    "parsed_columns": summary.get("schema", {}).get("columns", []),
    "available_variables": summary.get("available_variables", summary.get("schema", {}).get("columns", [])),
    "column_mapping": summary.get("column_mapping", {}),
}
inventory_path.write_text(json.dumps(inventory, indent=2), encoding="utf-8")
if summary.get("raw_file"):
    raw_df, _ = load_pangaea_dataset()
    print(raw_df.head())


{
  "dataset": {
    "name": "pangaea",
    "label": "PANGAEA 915062",
    "source_url": "https://doi.org/10.1594/PANGAEA.915062",
    "raw_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\data\\pangaea",
    "results_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\results\\pangaea",
    "notebooks_dir": "C:\\Users\\carla\\Desktop\\EECE 798K\\Project\\notebooks\\pangaea",
    "preferred_raw_names": [
      "PANGAEA.915062.txt",
      "pangaea_915062.txt",
      "pangaea_915062.tsv",
      "pangaea_915062.tab"
    ],
    "raw_globs": [
      "*.txt",
      "*.tab",
      "*.tsv",
      "*.csv"
    ],
    "column_aliases": {
      "time": [
        "time",
        "t"
      ],
      "tau": [
        "tau",
        "shear stress",
        "shear_stress",
        "stress",
        "friction",
        "mu",
        "\u03bc",
        "\u00b5"
      ],
      "displacement": [
        "u",
        "displacement",
        "slip",
        "shear displacement"
      ],
      "velocity"

In [3]:
if summary.get("raw_file"):
    numeric_cols = raw_df.select_dtypes(include=["number"]).columns.tolist()
    if summary.get("column_mapping", {}).get("time") and summary.get("column_mapping", {}).get("tau"):
        preview_df = raw_df[[summary["column_mapping"]["time"], summary["column_mapping"]["tau"]]].copy()
        preview_df.columns = ["time", "tau"]
        plot_signal_panel(preview_df.head(min(2000, len(preview_df))), "time", ["tau"], "PANGAEA primary signal preview", figsize=(12, 4))
    elif len(numeric_cols) >= 2:
        preview_df = raw_df[numeric_cols[:2]].head(min(2000, len(raw_df))).copy()
        preview_df.insert(0, "sample", range(len(preview_df)))
        plot_signal_panel(preview_df, "sample", numeric_cols[:2], "PANGAEA numeric preview")
    else:
        print("No numeric preview available for plotting.")
else:
    print("Raw file missing. See dataset_summary.json for the exact file placement instructions.")
